# Spatial Crop Modeler

Ask for a yield map in plain language — *"potential yield for beans in San Isidro, Honduras, 2020"* —
and get one. A Gemini agent loads the [`spatial-crop-modeler`](https://github.com/CGIAR-Climate-Data-Hub/skills)
skill, works out the bounding box, crop, cultivar and planting windows with you, and then this
notebook does the heavy lifting: download CHIRPS rainfall and CHIRTS temperature, add solar
radiation, build the weather cube, download SoilGrids, build the soil cube, run DSSAT pixel by
pixel, and draw the maps.

---

### How to use this notebook

1. **Runtime ▸ Run all.** Setup takes about five minutes — DSSAT and the geospatial stack are big.
2. Edit the request in section **5** if you want a different place, crop or year, then answer the
   agent's questions in the chat box. Pressing Enter on an empty box accepts what it proposes.
3. The simulation runs in section **5** too, and the maps appear in section **6**.

Most cells are collapsed to a single title bar — click ▸ to read the code. The cells you may want
to change are marked **✏️ EDIT ME**.

### Before you start

| Requirement | How |
|---|---|
| Gemini API key | Create one at [aistudio.google.com](https://aistudio.google.com/apikey), then add it in Colab under **Secrets** as `GOOGLE_API_KEY` with notebook access enabled. |
| Time | A municipality-sized area for one year takes roughly **20–50 minutes** end to end. Keep the tab open; Colab disconnects idle sessions. |
| Nothing else | CHIRPS, CHIRTS, NASA POWER and SoilGrids are all open. No second API key needed. |

> **Solar radiation without a key.** The skill's default for `rsds` is AgERA5, which needs a
> Copernicus CDS account. This notebook defaults to **NASA POWER** instead — no key, but 0.5°
> rather than 0.1°, so radiation is uniform across a small area. If you have a `~/.cdsapirc`,
> tell the agent to use `agera5` and it will.

> **Re-runs are cheap.** Every artefact is cached on disk. Ask for the same area twice and the
> download and cube-building steps report *already present* and skip straight to the simulation.


---
## 1 · Setup

Installs the stack, downloads the skill, and connects to Gemini. The install cell is the slow one.


In [ ]:
#@title Install ag-cube-cm, aggeodata and the Gemini SDK { display-mode: "form" }
# ag-cube-cm[models] pulls DSSATTools, which ships the DSSAT Fortran binaries.
# This takes ~4-5 minutes. Restart the runtime only if a cell below reports an import error.
%pip install -q "ag-cube-cm[models,viz] @ git+https://github.com/CGIAR-Climate-Data-Hub/ag-cube-cm.git"
%pip install -q "aggeodata[download] @ git+https://github.com/CGIAR-Climate-Data-Hub/aggeodata.git"
%pip install -q -U "google-genai>=2.9.0"
print("Install finished.")

In [ ]:
#@title Environment check - Step 0 of the skill { display-mode: "form" }
import importlib.metadata as md_, sys

missing = []
for pkg in ["ag-cube-cm", "aggeodata", "DSSATTools"]:
    try:
        print(f"OK       {pkg} {md_.version(pkg)}")
    except md_.PackageNotFoundError:
        print(f"MISSING  {pkg}")
        missing.append(pkg)

if missing:
    raise SystemExit(
        f"Missing: {', '.join(missing)}. Re-run the install cell above; if it still fails, "
        "use Runtime > Restart session and run all."
    )

# DSSAT is a Fortran program that silently fails on paths containing spaces.
# Everything this notebook writes lives under /content, which has none.
import pathlib
BASE       = pathlib.Path("/content/crop_run")
DSSAT_WORK = pathlib.Path("/content/dssat_work")
for d in (BASE, DSSAT_WORK):
    d.mkdir(parents=True, exist_ok=True)
print(f"\nWorking dirs: {BASE}  {DSSAT_WORK}")

In [ ]:
#@title Download the `spatial-crop-modeler` skill { display-mode: "form" }
import pathlib

!npx -y skills add https://github.com/cgiar-climate-data-hub/skills --skill spatial-crop-modeler --agent antigravity -y

SKILL_ROOT = pathlib.Path("/content/.agents/skills")
SKILL_NAME = "spatial-crop-modeler"
MOUNT      = "/content/.agents/skills"          # where the skill is mounted inside the sandbox
SKILL_DIR  = f"{MOUNT}/{SKILL_NAME}"
SKILL_MD   = SKILL_ROOT / SKILL_NAME / "SKILL.md"

assert SKILL_MD.exists(), "Skill download failed - re-run this cell."
SKILL_TEXT = SKILL_MD.read_text(encoding="utf-8")
print(f"Skill ready: {SKILL_MD}  ({len(SKILL_TEXT):,} chars)")

In [ ]:
#@title Connect to the Gemini API { display-mode: "form" }
SECRET_NAME = "GOOGLE_API_KEY" #@param {type:"string"}

import uuid, json
from google import genai
from google.colab import userdata
from IPython.display import display, Markdown
import ipywidgets as widgets

client = genai.Client(api_key=userdata.get(SECRET_NAME))
AGENT  = "antigravity-preview-05-2026"
print("Connected.")

---
## 2 · The pipeline

The six steps the skill orchestrates, as one function. Each step checks for its own output first,
so **an interrupted or repeated run resumes rather than re-downloading** — that is what makes
*"if the climate data isn't there, download it"* cheap to ask for twice.

| Step | Does | Cached artefact |
|------|------|-----------------|
| 1 | Write the aggeodata download config | `aggeodata_<suffix>.yaml` |
| 2 | Download CHIRPS `pr`, CHIRTS `tasmax`/`tasmin`, `rsds` | `climate_raw/` |
| 3 | Stack them into a weather cube | `climate_<suffix>_<y0>_<y1>.nc` |
| 4 | Download SoilGrids tiles | `soil_raw/*.tif` |
| 5 | Build the soil cube | `soil_<suffix>.nc` |
| 6 | Run DSSAT over every pixel | `yield_<suffix>.nc` |

Two details that decide whether a run produces anything at all:

- **The soil CRS.** `SoilDataCubeBuilder` defaults to `ESRI:54052`, which yields a cube with
  coordinates in metres and no overlap with the climate grid — every pixel then skips. SoilGrids'
  native projection is passed explicitly as `+proj=igh` and reprojected to `EPSG:4326`.
- **`ncores: 1` for the download.** CHIRPS and CHIRTS go through a writer that is not
  parallel-safe.


In [ ]:
#@title Pipeline functions - download, build cubes, simulate { display-mode: "form" }
import json, time, traceback
import yaml
import numpy as np
import xarray as xr

SOIL_VARS   = ["clay", "sand", "silt", "bdod", "cfvo", "soc", "phh2o", "wv0010", "wv0033", "wv1500"]
SOIL_DEPTHS = ["0-5", "5-15", "15-30", "30-60", "60-100"]
IGH = "+proj=igh +lat_0=0 +lon_0=0 +datum=WGS84 +units=m +no_defs"   # SoilGrids native

# Yield floor for the quality gate, per crop. The skill's default of 200 kg/ha is tuned for
# maize; legumes legitimately yield less, so beans get a lower bar.
MIN_MEAN_HWAM = {"Bean": 150.0, "Maize": 200.0, "Wheat": 200.0}

state = {}          # filled by the chat plumbing in the next cell; holds the last run


def _say(msg):
    """Progress line, shown in the chat output area when there is one."""
    try:
        with out:
            print(msg)
    except NameError:
        print(msg)


def run_pipeline(bbox, start_date, end_date, crop, cultivar, planting_date,
                 suffix="run", n_planting_windows=4, planting_window_days=14,
                 rsds_source="nasa_power", max_pixels=None,
                 country="Honduras", country_code="HND", **_):
    """Download what is missing, build the cubes, run DSSAT, and gate the result."""
    try:
        return _run_pipeline(bbox, start_date, end_date, crop, cultivar, planting_date,
                             suffix, n_planting_windows, planting_window_days,
                             rsds_source, max_pixels, country, country_code)
    except Exception as exc:
        _say(f"\nPipeline error: {exc}")
        return {"state": "error", "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc()[-1500:]}


def _run_pipeline(bbox, start_date, end_date, crop, cultivar, planting_date,
                  suffix, n_planting_windows, planting_window_days,
                  rsds_source, max_pixels, country, country_code):
    t0   = time.time()
    run  = BASE / suffix
    clim = run / "climate_raw"
    soil = run / "soil_raw"
    for d in (run, clim, soil):
        d.mkdir(parents=True, exist_ok=True)

    y0, y1     = start_date[:4], end_date[:4]
    weather_nc = clim / f"climate_{suffix}_{y0}_{y1}.nc"
    soil_nc    = run / f"soil_{suffix}.nc"
    yield_nc   = run / f"yield_{suffix}.nc"
    ageo_cfg   = run / f"aggeodata_{suffix}.yaml"
    sim_cfg    = run / f"dssat_{suffix}.yaml"
    steps      = {}

    # -- 1. aggeodata download config -----------------------------------------
    cfg = {
        "dates":        {"starting_date": start_date, "ending_date": end_date},
        "spatial_info": {"spatial_file": None, "extent": [float(v) for v in bbox]},
        "climate":      {"variables": {"pr":     {"source": "chirps"},
                                       "tasmax": {"source": "chirts"},
                                       "tasmin": {"source": "chirts"},
                                       "rsds":   {"source": rsds_source}}},
        "soil":         {"enabled": False},
        "general":      {"suffix": suffix, "ncores": 1, "task": "download",
                         "reference_variable": "pr", "agera5_version": "2_0",
                         "target_crs": "EPSG:4326"},
        "paths":        {"output_path": str(clim)},
    }
    ageo_cfg.write_text(yaml.dump(cfg, sort_keys=False), encoding="utf-8")
    steps["config"] = f"written {ageo_cfg.name}"
    _say(f"[1/6] config -> {ageo_cfg.name}")

    # -- 2. climate download ---------------------------------------------------
    already = [p for p in clim.glob("*_raw") if any(p.rglob("*.nc")) or any(p.rglob("*.tif"))]
    if already:
        steps["climate_download"] = f"cached ({len(already)} variable folders present)"
        _say(f"[2/6] climate download - already present, skipping")
    else:
        _say(f"[2/6] climate download - CHIRPS + CHIRTS + {rsds_source} ... (several minutes)")
        from aggeodata.pipelines.download import run_download
        res = run_download(str(ageo_cfg))
        n = sum(len(v) for v in res.values()) if isinstance(res, dict) else 0
        steps["climate_download"] = f"downloaded {n} files"
        _say(f"      {n} files")

    # -- 3. weather cube -------------------------------------------------------
    if weather_nc.exists():
        steps["weather_cube"] = f"cached {weather_nc.name}"
        _say(f"[3/6] weather cube - already exists")
    else:
        _say("[3/6] weather cube - stacking ...")
        from aggeodata.pipelines.datacube import run_datacube
        weather_nc = pathlib.Path(run_datacube(str(ageo_cfg)))
        steps["weather_cube"] = f"built {weather_nc.name}"
    with xr.open_dataset(weather_nc) as wds:
        steps["weather_shape"] = {"vars": list(wds.data_vars), "dims": {k: int(v) for k, v in wds.sizes.items()}}
    _say(f"      {steps['weather_shape']}")

    # -- 4. soil download ------------------------------------------------------
    tifs = list(soil.glob("*.tif"))
    if tifs:
        steps["soil_download"] = f"cached ({len(tifs)} tiles)"
        _say(f"[4/6] soil download - already present ({len(tifs)} tiles)")
    else:
        _say("[4/6] soil download - SoilGrids ...")
        from aggeodata.ingestion.soil import SoilGridsDownloader
        got = SoilGridsDownloader(soil_layers=SOIL_VARS, depths=SOIL_DEPTHS,
                                  output_folder=str(soil)).download(boundaries=list(bbox))
        steps["soil_download"] = f"downloaded {len(got)} tiles"
        _say(f"      {len(got)} tiles")

    # -- 5. soil cube ----------------------------------------------------------
    if soil_nc.exists():
        steps["soil_cube"] = f"cached {soil_nc.name}"
        _say("[5/6] soil cube - already exists")
    else:
        _say("[5/6] soil cube - building (explicit +proj=igh -> EPSG:4326) ...")
        from aggeodata.transform.soil_cube import SoilDataCubeBuilder
        SoilDataCubeBuilder(
            data_folder=str(soil), variables=SOIL_VARS,
            reference_variable="wv1500", crs=IGH, target_crs="EPSG:4326",
        ).build_and_save(output_path=str(run), filename=soil_nc.name)
        steps["soil_cube"] = f"built {soil_nc.name}"

    sds = xr.open_dataset(soil_nc)
    flat = [v for v in sds.data_vars if "_cm_mean" in v]
    if flat:                                   # older pipelines emit one var per depth
        from aggeodata.transform.soil_cube import reshape_flat_soil_cube
        sds.close()
        reshape_flat_soil_cube(xr.open_dataset(soil_nc)).to_netcdf(soil_nc)
        sds = xr.open_dataset(soil_nc)
        steps["soil_cube"] += " (reshaped from flat format)"
    assert abs(float(sds.y.min())) < 90 and abs(float(sds.x.min())) < 180, \
        "Soil cube is in projected metres - rebuild with crs=IGH and target_crs='EPSG:4326'."
    sds.close()

    # -- 6. DSSAT --------------------------------------------------------------
    sim = {
        "GENERAL_INFO": {"country": country, "country_code": country_code, "model": "dssat",
                         "working_path": str(DSSAT_WORK), "dssat_path": None, "ncores": 2},
        "SPATIAL_INFO": {"feature_name": "shapeName", "adm_level": 2, "feature": None,
                         "weather_path": str(weather_nc), "soil_path": str(soil_nc),
                         "output_path": str(yield_nc), "dem_path": None},
        "CROP":       {"name": crop, "cultivar": cultivar},
        "MANAGEMENT": {"planting_date": planting_date,
                       "n_planting_windows": int(n_planting_windows),
                       "planting_window_days": int(planting_window_days)},
    }
    sim_cfg.write_text(yaml.dump(sim, sort_keys=False), encoding="utf-8")

    if yield_nc.exists():
        steps["simulation"] = f"cached {yield_nc.name}"
        _say("[6/6] DSSAT - output already exists, skipping")
        sim_result = {"status": "ok", "output_path": str(yield_nc), "cached": True}
    else:
        _say(f"[6/6] DSSAT - {crop} {cultivar}, {n_planting_windows} planting window(s) ...")
        from ag_cube_cm.mcp_server import run_simulation
        sim_result = json.loads(run_simulation(config_path=str(sim_cfg), max_pixels=max_pixels))
        if sim_result.get("status") != "ok":
            _say(f"      DSSAT reported: {sim_result.get('message')}")
            return {"state": "simulation_failed", "message": sim_result.get("message"),
                    "steps": steps, "config": str(sim_cfg)}
        steps["simulation"] = (f"ok={sim_result.get('pixels_ok')} "
                               f"skipped={sim_result.get('pixels_skipped')} "
                               f"failed={sim_result.get('pixels_failed')}")

    gate = quality_gate(yield_nc, crop)
    state.update(yield_nc=yield_nc, weather_nc=weather_nc, soil_nc=soil_nc,
                 crop=crop, cultivar=cultivar, suffix=suffix,
                 planting_date=planting_date, planting_window_days=int(planting_window_days))

    _say(f"\nPipeline finished in {(time.time() - t0) / 60:.1f} min - "
         f"quality gate {gate['verdict']}")
    return {"state": "ok", "steps": steps, "quality_gate": gate,
            "paths": {"weather": str(weather_nc), "soil": str(soil_nc), "yield": str(yield_nc)},
            "simulation": {k: v for k, v in sim_result.items() if k != "message"}}


def quality_gate(yield_path, crop="Maize", **_):
    """Step 7 of the skill. A clean DSSAT exit is not a quality signal."""
    ds = xr.open_dataset(str(yield_path))
    total     = int(ds.flag.size)
    n_ok      = int((ds.flag.values == 0).sum())
    n_failed  = int((ds.flag.values == 1).sum())
    n_no_data = int((ds.flag.values == 2).sum())
    mean_hwam = float(np.nanmean(ds.HWAM.where(ds.flag == 0).values)) if n_ok else 0.0
    ds.close()

    ok_frac   = n_ok / total if total else 0.0
    fail_frac = n_failed / max(n_ok + n_failed, 1)
    floor     = MIN_MEAN_HWAM.get(crop, 200.0)
    checks = {
        "ok_fraction":   (round(ok_frac, 3),   ok_frac   >= 0.20),
        "fail_fraction": (round(fail_frac, 3), fail_frac <= 0.50),
        "mean_hwam":     (round(mean_hwam, 1), mean_hwam >= floor),
    }
    passed = all(p for _, p in checks.values())
    return {
        "verdict": "PASSED" if passed else "FAILED",
        "pixels": {"ok": n_ok, "failed": n_failed, "no_data": n_no_data, "total": total},
        "mean_hwam_kg_ha": round(mean_hwam, 1), "hwam_floor_used": floor, "checks": checks,
        "diagnosis": None if passed else (
            "Probable causes, ordered: (1) bbox over water - soil and rsds are NaN there; "
            "(2) wrong planting season for this region; (3) soil/climate grid mismatch; "
            "(4) cultivar incompatible with the climate."),
    }


print("Pipeline loaded.")

---
## 3 · Agent tools

Four functions run **here in the notebook**, not in the agent's sandbox — the sandbox has no
keyboard, no open internet, no DSSAT and none of your data:

- **`ask_user`** — poses a batch of questions and returns every answer together.
- **`resolve_place`** — turns *"San Isidro, Yoro, Honduras"* into a bounding box via
  OpenStreetMap, and pads anything smaller than a few CHIRPS pixels.
- **`run_pipeline`** — the six steps above.
- **`plot_maps`** — draws the multitemporal panels.

**Why answering is free.** The tools are *batched*: the agent sends a dozen questions in one turn,
you answer them one at a time locally, and the notebook spends a single API call sending every
answer back. Questions cost nothing — **turns** cost. The cell also backs off and retries when the
free tier returns a `429`.

### Why the bbox gets padded

CHIRPS and CHIRTS are 0.05° grids — about 5.5 km. An OSM *village* polygon is often 0.04° across,
which is **less than one pixel**, and the outermost rows and columns of a downloaded cube are
usually NaN anyway. `resolve_place` therefore grows any box below `min_span_deg` (default 0.3°,
roughly 6×6 pixels) around its centre and says so. Without that, DSSAT gets one usable pixel or
none, and the quality gate fails for a reason that has nothing to do with agronomy.


In [ ]:
#@title Tool functions - `resolve_place` and the tool declarations { display-mode: "form" }
import urllib.request, urllib.parse, urllib.error, socket, re

UA          = {"User-Agent": "cdh-spatial-crop-modeler-notebook/1.0"}
NOMINATIM   = "https://nominatim.openstreetmap.org/search"
MAX_RETRIES = 4


def resolve_place(place, min_span_deg=0.3, limit=8, **_):
    """Geocode a place to [xmin, ymin, xmax, ymax], padding boxes too small to model.

    Falls back to searching the bare place name within the country when the full
    "town, department, country" string finds nothing - misremembered departments are
    common, and the candidate list is more useful than a flat failure.
    """
    def search(q):
        url = NOMINATIM + "?" + urllib.parse.urlencode(
            {"q": q, "format": "jsonv2", "limit": limit, "addressdetails": 1})
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=30) as r:
                return json.loads(r.read())
        except (urllib.error.URLError, socket.timeout, ValueError, OSError) as exc:
            return {"error": str(exc)}

    hits = search(place)
    if isinstance(hits, dict):
        return {"state": "unavailable", "note": f"OpenStreetMap unreachable: {hits['error']}"}

    note  = None
    parts = [p.strip() for p in place.split(",") if p.strip()]
    if not hits and len(parts) >= 3:
        # drop the middle admin level, keep "name, country"
        broader = f"{parts[0]}, {parts[-1]}"
        hits = search(broader)
        if isinstance(hits, dict):
            return {"state": "unavailable", "note": f"OpenStreetMap unreachable: {hits['error']}"}
        if hits:
            note = (f"No match for '{place}'. These are the '{parts[0]}' places in "
                    f"{parts[-1]} - ask the user which one they mean, naming the region of each.")

    if not hits:
        return {"state": "not_found", "query": place,
                "note": "Nothing matched. Ask the user for the region, a nearby larger town, "
                        "or an explicit bbox."}

    candidates = []
    for h in hits:
        south, north, west, east = (float(v) for v in h["boundingbox"])
        bbox = [west, south, east, north]
        span = max(east - west, north - south)
        padded = False
        if span < min_span_deg:                      # grow around the centre
            cx, cy = (west + east) / 2, (south + north) / 2
            half   = min_span_deg / 2
            bbox   = [round(cx - half, 4), round(cy - half, 4),
                      round(cx + half, 4), round(cy + half, 4)]
            padded = True
        addr = h.get("address", {})
        candidates.append({
            "name":   h.get("name"),
            "kind":   h.get("type"),
            "region": addr.get("state") or addr.get("county") or addr.get("region"),
            "country": addr.get("country"),
            "bbox":   [round(v, 4) for v in bbox],
            "pixels_at_0.05deg": int(round((bbox[2] - bbox[0]) / 0.05)) ** 2,
            "padded_to_min_span": padded,
        })
    return {"state": "ok", "query": place, "candidates": candidates, "note": note}


# --- tool 1: interview the user ----------------------------------------------
ask_user_tool = {
    "type": "function",
    "name": "ask_user",
    "description": (
        "Ask the user for a BATCH of parameters in a single call and receive every answer "
        "together. Put EVERY field you still need into `questions` - the client presents them "
        "one at a time and returns all the answers in one response, so a batch of 10 costs "
        "exactly what a batch of 1 costs. Calling this with one question and waiting burns the "
        "user's quota and is the most common way this workflow fails."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "questions": {
                "type": "array",
                "description": "Every parameter you need right now. Longer is better.",
                "items": {
                    "type": "object",
                    "properties": {
                        "field":      {"type": "string", "description": "Parameter, e.g. 'planting_date'."},
                        "question":   {"type": "string", "description": "The question, in plain language."},
                        "suggestion": {"type": "string", "description": "Draft value the user accepts with Enter."},
                    },
                    "required": ["field", "question"],
                },
            },
        },
        "required": ["questions"],
    },
}

# --- tool 2: geocode ----------------------------------------------------------
resolve_place_tool = {
    "type": "function",
    "name": "resolve_place",
    "description": (
        "Turn a place name into a bounding box using OpenStreetMap. Runs on the user's machine; "
        "your sandbox has no internet. Pass the fullest name you have, e.g. "
        "'San Isidro, Yoro, Honduras'. Returns ranked candidates, each with its region, so you "
        "can disambiguate - many Latin American place names repeat across departments. Boxes "
        "smaller than a few CHIRPS pixels are padded and flagged with `padded_to_min_span`. "
        "Always show the chosen candidate's bbox to the user for confirmation before simulating; "
        "never invent coordinates yourself."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "place":        {"type": "string", "description": "Place name, ideally 'town, region, country'."},
            "min_span_deg": {"type": "number", "description": "Smallest acceptable bbox span in degrees (default 0.3)."},
        },
        "required": ["place"],
    },
}

# --- tool 3: the pipeline -----------------------------------------------------
run_pipeline_tool = {
    "type": "function",
    "name": "run_pipeline",
    "description": (
        "Download any missing climate and soil data, build the datacubes, run DSSAT over every "
        "pixel, and apply the skill's quality gate. Runs in the notebook - your sandbox has "
        "neither the data nor the DSSAT binary. Steps whose output already exists are skipped, "
        "so calling this twice for the same suffix is cheap. Takes 20-50 minutes on a fresh "
        "area, so call it ONCE, only after the user has confirmed the plan. Returns the pixel "
        "summary and the quality gate verdict; if the gate FAILED, report the diagnosis and stop "
        "rather than plotting."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "bbox":        {"type": "array", "items": {"type": "number"},
                            "description": "[xmin, ymin, xmax, ymax] in WGS-84 degrees."},
            "start_date":  {"type": "string", "description": "YYYY-MM-DD, start of the weather record."},
            "end_date":    {"type": "string", "description": "YYYY-MM-DD, end of the weather record."},
            "crop":        {"type": "string", "description": "Bean, Maize or Wheat."},
            "cultivar":    {"type": "string", "description": "DSSAT cultivar code, e.g. IB0001 for Bean."},
            "planting_date": {"type": "string", "description": "YYYY-MM-DD, first planting window."},
            "n_planting_windows":   {"type": "integer", "description": "Planting windows to simulate (4 gives four maps)."},
            "planting_window_days": {"type": "integer", "description": "Days between windows (default 14)."},
            "suffix":      {"type": "string", "description": "Short label, no spaces, e.g. 'beans_sanisidro_2020'."},
            "rsds_source": {"type": "string", "description": "'nasa_power' (no key) or 'agera5' (needs ~/.cdsapirc)."},
            "max_pixels":  {"type": "integer", "description": "Cap for a quick test; omit to run every pixel."},
            "country":      {"type": "string", "description": "Country name, e.g. Honduras."},
            "country_code": {"type": "string", "description": "ISO3, e.g. HND."},
        },
        "required": ["bbox", "start_date", "end_date", "crop", "cultivar",
                     "planting_date", "suffix"],
    },
}

# --- tool 4: the maps ---------------------------------------------------------
plot_maps_tool = {
    "type": "function",
    "name": "plot_maps",
    "description": (
        "Draw the multitemporal maps in the notebook after the quality gate passes. "
        "kind='yield' gives one panel per planting window (and per year, when several were "
        "simulated); kind='climate' gives monthly rainfall and max-temperature panels from the "
        "weather cube; kind='both' draws both. Call it once, at the end. Returns the numbers "
        "behind each panel so you can describe them - you cannot see the images."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "kind": {"type": "string", "description": "'yield', 'climate' or 'both' (default)."},
        },
        "required": [],
    },
}

TOOLS = [ask_user_tool, resolve_place_tool, run_pipeline_tool, plot_maps_tool]
print("Tools declared:", [t["name"] for t in TOOLS])

In [ ]:
#@title Plotting - multitemporal yield and climate panels { display-mode: "form" }
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 10, "font.size": 9})


def _panels(n):
    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.4 * cols, 3.1 * rows), squeeze=False)
    return fig, [a for row in axes for a in row]


def plot_yield_maps():
    """One map per planting window per year, plus the mean across all of them."""
    ds   = xr.open_dataset(str(state["yield_nc"]))
    hwam = ds["HWAM"].where(ds["flag"] == 0)          # flag 0 = DSSAT produced output
    pw   = [int(v) for v in np.atleast_1d(hwam["planting_window"].values)]
    yrs  = [int(v) for v in np.atleast_1d(hwam["year"].values)]

    frames, stats = [], []
    for y in yrs:
        for w in pw:
            sl  = hwam.sel(year=y, planting_window=w)
            day = (np.datetime64(state["planting_date"])
                   + np.timedelta64(w * state["planting_window_days"], "D"))
            vals = sl.values[~np.isnan(sl.values)]
            frames.append((f"{state['crop']} {y}\nplanted {str(day)[:10]}", sl))
            stats.append({"year": y, "planting_window": w, "planting_date": str(day)[:10],
                          "pixels_with_yield": int(vals.size),
                          "mean_kg_ha": round(float(vals.mean()), 1) if vals.size else None,
                          "min_kg_ha":  round(float(vals.min()), 1)  if vals.size else None,
                          "max_kg_ha":  round(float(vals.max()), 1)  if vals.size else None})

    allv = hwam.values[~np.isnan(hwam.values)]
    vmax = float(np.nanpercentile(allv, 98)) if allv.size else 1.0

    fig, axes = _panels(len(frames) + 1)
    for ax, (title, sl) in zip(axes, frames):
        sl.plot(ax=ax, cmap="YlGn", vmin=0, vmax=vmax, add_colorbar=False)
        ax.set_title(title); ax.set_xlabel(""); ax.set_ylabel("")
    mean = hwam.mean(dim=["planting_window", "year"])
    im   = mean.plot(ax=axes[len(frames)], cmap="YlGn", vmin=0, vmax=vmax, add_colorbar=False)
    axes[len(frames)].set_title("mean of all windows"); axes[len(frames)].set_xlabel(""); axes[len(frames)].set_ylabel("")
    for ax in axes[len(frames) + 1:]:
        ax.axis("off")
    fig.colorbar(im, ax=axes, shrink=0.7, label="HWAM (kg/ha)")
    fig.suptitle(f"{state['crop']} {state['cultivar']} yield - {state['suffix']}", y=1.02)
    png = state["yield_nc"].parent / f"yield_maps_{state['suffix']}.png"
    fig.savefig(png, bbox_inches="tight"); plt.show()
    ds.close()
    return {"panels": stats, "png": str(png),
            "colour_scale_max_kg_ha": round(vmax, 1)}


def plot_climate_maps():
    """Monthly rainfall total and mean daily max temperature from the weather cube."""
    ds = xr.open_dataset(str(state["weather_nc"]))
    summaries = {}
    for var, how, cmap, label in [("pr", "sum", "Blues", "monthly rainfall (mm)"),
                                  ("tasmax", "mean", "YlOrRd", "mean daily Tmax (degC)")]:
        if var not in ds.data_vars:
            continue
        monthly = getattr(ds[var].resample(time="1MS"), how)()
        fig, axes = _panels(int(monthly.sizes["time"]))
        vmin = float(np.nanpercentile(monthly.values, 2))
        vmax = float(np.nanpercentile(monthly.values, 98))
        im = None
        for ax, t in zip(axes, monthly["time"].values):
            im = monthly.sel(time=t).plot(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, add_colorbar=False)
            ax.set_title(str(t)[:7]); ax.set_xlabel(""); ax.set_ylabel("")
        for ax in axes[int(monthly.sizes["time"]):]:
            ax.axis("off")
        if im is not None:
            fig.colorbar(im, ax=axes, shrink=0.7, label=label)
        fig.suptitle(f"{label} - {state['suffix']}", y=1.02)
        png = state["weather_nc"].parent / f"climate_{var}_{state['suffix']}.png"
        fig.savefig(png, bbox_inches="tight"); plt.show()
        summaries[var] = {
            "by_month": {str(t)[:7]: round(float(np.nanmean(monthly.sel(time=t).values)), 1)
                         for t in monthly["time"].values},
            "png": str(png)}
    ds.close()
    return summaries


def plot_maps(kind="both", **_):
    if not state.get("yield_nc"):
        return {"state": "nothing_to_plot", "note": "run_pipeline has not produced a result yet"}
    result = {"state": "ok"}
    if kind in ("both", "climate"):
        result["climate"] = plot_climate_maps()
    if kind in ("both", "yield"):
        result["yield"] = plot_yield_maps()
    return result


print("Plotting loaded.")

In [ ]:
#@title Chat plumbing - the question box and the turn loop { display-mode: "form" }
out  = widgets.Output()
box  = widgets.Text(layout=widgets.Layout(width="80%"))
send = widgets.Button(description="Send", button_style="primary")
ui   = widgets.HBox([box, send])

state.update({"result": None, "env": None, "queue": [], "answers": [],
              "subq": None, "collected": {}, "batch": 0, "turns": 0})

LOCAL = {"resolve_place": resolve_place, "run_pipeline": run_pipeline, "plot_maps": plot_maps}


def api_call(**kwargs):
    """interactions.create with backoff, so a free-tier 429 pauses instead of crashing."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            res = client.interactions.create(**kwargs)
            state["turns"] += 1
            return res
        except Exception as exc:
            text = str(exc)
            if "429" not in text and "too_many_requests" not in text.lower():
                raise
            found = re.search(r"retry in ([\d.]+)s", text)
            wait  = float(found.group(1)) + 2 if found else 20 * attempt
            with out:
                print(f"rate limited - waiting {wait:.0f}s "
                      f"(attempt {attempt}/{MAX_RETRIES}; the notebook will resume on its own)")
            time.sleep(wait)
    raise RuntimeError("Still rate limited after several retries - wait a minute and re-run, "
                       "or use a paid key.")


def show(obj, prefix="**Agent**\n\n"):
    text = obj if isinstance(obj, str) else getattr(obj, "output_text", None)
    if text:
        display(Markdown(prefix + text))


def pending_calls(res):
    done = {s.call_id for s in res.steps if s.type == "function_result"}
    return [s for s in res.steps if s.type == "function_call" and s.id not in done]


def call_args(call):
    return call.arguments if isinstance(call.arguments, dict) else json.loads(call.arguments)


def reply(call, payload):
    state["answers"].append({"type": "function_result", "name": call.name,
                             "call_id": call.id, "result": payload})


def end(msg):
    box.disabled = send.disabled = True
    with out:
        display(Markdown(f"### {msg}"))


def run_local(call):
    """Execute a non-interactive tool call here in the notebook."""
    fn = LOCAL.get(call.name)
    if fn is None:
        return {"error": f"no local handler for {call.name}"}
    with out:
        display(Markdown(f"*running `{call.name}`...*"))
    res = fn(**call_args(call))
    with out:
        if call.name == "resolve_place" and res.get("candidates"):
            display(Markdown("\n".join(
                f"- **{c['name']}** ({c['kind']}, {c['region']}) `{c['bbox']}` "
                f"~{c['pixels_at_0.05deg']} px" + ("  *padded*" if c["padded_to_min_span"] else "")
                for c in res["candidates"][:8])))
        elif call.name == "run_pipeline" and res.get("quality_gate"):
            g = res["quality_gate"]
            display(Markdown(
                f"**Quality gate {g['verdict']}** - pixels ok {g['pixels']['ok']}/"
                f"{g['pixels']['total']}, failed {g['pixels']['failed']}, "
                f"no_data {g['pixels']['no_data']}, mean HWAM {g['mean_hwam_kg_ha']} kg/ha"))
    return res


def questions_of(call):
    args  = call_args(call)
    items = args.get("questions")
    if isinstance(items, list) and items:
        return [q for q in items if isinstance(q, dict) and q.get("question")]
    if args.get("question"):
        return [{"field": args.get("field", "value"), "question": args["question"],
                 "suggestion": args.get("suggestion")}]
    return []


def pose(q):
    done    = state["batch"] - len(state["subq"]) + 1
    counter = f"`{done}/{state['batch']}` " if state["batch"] > 1 else ""
    text    = f"{counter}**{q.get('field', 'question')}** - {q['question']}"
    if q.get("suggestion"):
        text += (f"\n\n> *Proposed:* {q['suggestion']}\n>\n"
                 "> *Press Enter on an empty box to accept, or type a replacement.*")
    with out:
        display(Markdown(text))
    box.placeholder = "Enter to accept" if q.get("suggestion") else "Your answer..."
    box.disabled = send.disabled = False
    box.value = ""


def advance_turn():
    box.disabled = send.disabled = True
    with out:
        print(f"thinking... (turn {state['turns'] + 1}, sending {len(state['answers'])} result(s))")
    state["result"] = api_call(agent=crop_agent.id,
                               previous_interaction_id=state["result"].id,
                               environment=state["env"], input=state["answers"], tools=TOOLS)
    state["answers"] = []
    with out:
        show(state["result"])
    ask_next()


def ask_next():
    """Drive the queue: run local tools silently, pause only for questions."""
    while True:
        while state["queue"] and state["queue"][0].name != "ask_user":
            call = state["queue"].pop(0)
            reply(call, run_local(call))

        if state["queue"]:
            call = state["queue"][0]
            if state["subq"] is None:
                state["subq"]      = questions_of(call)
                state["collected"] = {}
                state["batch"]     = len(state["subq"])
                if state["batch"] > 1:
                    with out:
                        display(Markdown(f"*{state['batch']} questions in this batch - answering "
                                         "all of them costs a single API turn.*"))
            if state["subq"]:
                return pose(state["subq"][0])
            state["queue"].pop(0)
            reply(call, {"answers": state["collected"]})
            state["subq"] = None
            continue

        if state["answers"]:
            return advance_turn()

        res = state["result"]
        if res.status != "requires_action":
            return end(f"Finished in {state['turns']} API turns.")
        calls = pending_calls(res)
        if not calls:
            return end(f"Finished in {state['turns']} API turns - nothing left to answer.")
        state["queue"] = calls


def submit(_):
    if not state["subq"] or box.disabled:
        return
    msg, box.value = box.value.strip(), ""
    q = state["subq"].pop(0)
    answer = msg or q.get("suggestion") or ""
    with out:
        display(Markdown(f"**You** - {answer}"))
    state["collected"][q.get("field", f"q{len(state['collected'])}")] = answer
    ask_next()


send.on_click(submit)
box.on_submit(submit)   # Enter key (deprecated in ipywidgets 8 - harmless warning)


def start(first_input):
    out.clear_output()
    state.update(result=None, env=None, queue=[], answers=[], subq=None,
                 collected={}, batch=0, turns=0)
    state["result"] = api_call(agent=crop_agent.id, input=first_input,
                               environment="remote", tools=TOOLS)
    state["env"] = state["result"].environment_id
    display(out, ui)
    with out:
        show(state["result"])
    ask_next()


print("Helpers loaded.")

---
## 4 · Agent instructions &nbsp; ✏️ **EDIT ME**

The agent's standing brief. The full `SKILL.md` is appended verbatim, so the agent never has to
spend a turn reading a file — and the operating rules above it say what to propose, what to ask,
and what never to guess. Widen the *PROPOSE* list to be asked fewer questions.


In [ ]:
BRIEF = """
You are a spatial crop modelling agent for the CGIAR Climate Data Hub. You run DSSAT over a
region and explain the result. The full spatial-crop-modeler skill is appended below - follow its
workflow, but note the four differences that apply in this notebook.

1. THE NOTEBOOK RUNS THE PIPELINE, NOT YOU. Your sandbox has no internet, no data and no DSSAT
   binary. Steps 0 and 3-6 of the skill are already implemented as the `run_pipeline` function:
   it writes both YAML configs, downloads whatever is missing, builds the weather and soil cubes
   (with the SoilGrids +proj=igh fix), runs DSSAT and applies the Step 7 quality gate. Do not
   write configs or shell commands yourself, and never ask the user to run anything.

2. YOU HAVE NO CHAT CHANNEL. The only way to ask the user anything is `ask_user`, which takes a
   LIST of questions. A question written as message text will never be answered, so never end a
   message with one. Batch everything: the user pays per turn, and every turn resends this whole
   conversation. Your FIRST ask_user call should carry every parameter you still need. Aim to
   finish in under six turns.

3. RESOLVE THE PLACE, DO NOT INVENT IT. Call `resolve_place` with the fullest name the user gave.
   It returns candidates with their region - if more than one plausibly matches, or the region the
   user named is missing from the results, ask them to choose; do not pick silently. Never write a
   bbox from memory. Always show the bbox you intend to use before simulating. When a candidate
   comes back with padded_to_min_span=true, tell the user the area was widened to get enough
   pixels to model.

4. MULTITEMPORAL MEANS PLANTING WINDOWS. One year and one planting window produce a single map.
   When the user asks to see yield over time within a single season, use n_planting_windows: 4
   with planting_window_days: 14 - four staggered sowings, four maps. Across years, widen the
   date range instead. State which you are doing.

PARAMETERS

  Propose, then confirm: bbox (from resolve_place), cultivar, planting_date, the number of
  planting windows, start_date/end_date (default: the whole calendar year requested), suffix
  (a short slug like beans_sanisidro_2020), country and country_code.

  Never guess: which candidate place the user meant when it is ambiguous, and any planting date
  for a region or cropping season you are not confident about. Ask, with your best proposal as
  the `suggestion` so a single Enter accepts it.

  Cultivars: Bean -> IB0001 (CRGRO048). Maize -> IB1072 tropical, PC0002 temperate.
  Wheat -> IB1015 winter, IB0001 spring. Wheat uses WHAPS048; WHCER codes such as IB1487 fail
  with rc=99.

  Solar radiation: rsds_source defaults to nasa_power, which needs no API key. Use agera5 only
  if the user says they have a CDS key.

  Planting dates in Central America: beans have two seasons - primera, sown May/June, and
  postrera, sown September. Ask which one the user means rather than assuming; it changes the
  result completely.

AFTER THE RUN

  Report the pixel summary verbatim: ok / failed / no_data and mean HWAM. If the quality gate
  FAILED, stop. Show the diagnosis, say which cause you think applies and why, and ask the user
  how to proceed - do not plot, and do not describe a failed run as a result. Only when the gate
  PASSES, call `plot_maps` once. You cannot see the images, so describe the numbers it returns:
  which planting window yielded most, the spread across the area, and where the low pixels are.
  Close by naming the three output files and the honest caveats - a potential-yield run with no
  fertiliser schedule, a coarse rsds grid if nasa_power was used, and a single cultivar.

--- FULL SKILL BELOW ---
"""

SYSTEM_INSTRUCTION = BRIEF + "\n" + SKILL_TEXT
print(f"System instruction: {len(SYSTEM_INSTRUCTION):,} chars "
      f"({len(BRIEF):,} brief + {len(SKILL_TEXT):,} skill)")

---
## 5 · Create the agent and make your request &nbsp; ✏️ **EDIT ME**

Re-run the agent cell after any edit to the instructions above — the brief is baked in when the
agent is created.

Then say what you want. A place name, a crop and a year is enough; the agent fills in the rest and
asks about anything it should not assume.

| You want | Write something like |
|---|---|
| A municipality | `potential yield for beans in San Isidro, Intibucá, Honduras for 2020` |
| A named bbox | `maize yield for bbox [-87.5, 14.2, -87.2, 14.5] in 2021` |
| Several years | `bean yield around Yoro, Honduras, 2018 to 2020` |


In [ ]:
#@title Create the agent { display-mode: "form" }
def skill_sources(root, skill, mount=MOUNT):
    """Mount the skill folder in the sandbox, so references stay available if needed."""
    src_dir, sources = root / skill, []
    for p in sorted(src_dir.rglob("*")):
        if not p.is_file() or "evals" in p.name:
            continue
        sources.append({"type": "inline",
                        "target": f"{mount}/{skill}/{p.relative_to(src_dir).as_posix()}",
                        "content": p.read_text(encoding="utf-8")})
    return sources


crop_agent = client.agents.create(
    id=f"spatial-crop-modeler-{uuid.uuid4().hex[:8]}",
    base_agent=AGENT,
    system_instruction=SYSTEM_INSTRUCTION,
    base_environment={"type": "remote", "sources": skill_sources(SKILL_ROOT, SKILL_NAME)},
)
print(f"Agent ready: {crop_agent.id}")

In [ ]:
PROMPT = "I would like to see the potential yield for beans in San Isidro, Yoro, Honduras for 2020" #@param {type:"string"}

start(PROMPT)

---
## 6 · The maps again

The agent draws the panels during the conversation above. Run this to redraw them — after a
reconnect, or to export the PNGs — without spending another API turn.


In [ ]:
#@title Redraw and list the outputs { display-mode: "form" }
if state.get("yield_nc"):
    summary = plot_maps("both")
    for panel in summary.get("yield", {}).get("panels", []):
        print(f"  {panel['planting_date']}  mean {panel['mean_kg_ha']} kg/ha "
              f"(range {panel['min_kg_ha']}-{panel['max_kg_ha']}, "
              f"{panel['pixels_with_yield']} pixels)")
    print("\nFiles:")
    for key in ("weather_nc", "soil_nc", "yield_nc"):
        print(f"  {key:11s} {state[key]}")
else:
    print("No simulation in this session yet - run the cells above first.")

---
## What next?

- **This is potential yield.** No fertiliser schedule, one cultivar, rainfed, and DSSAT's own
  assumptions about management. Treat the numbers as an upper envelope shaped by weather and soil,
  not a forecast of what a farm will harvest.
- **Check the failed pixels before trusting the mean.** A cluster of `flag=2` on one edge usually
  means water or a grid mismatch, not bad agronomy. The quality gate catches the catastrophic
  cases only.
- **Document the run** with the [`cdh-metadata` notebook](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/skills/blob/main/notebooks/CDH_metadata_example.ipynb)
  — it turns `yield_<suffix>.nc` into a CDH catalogue record.
- **Read the skill**: [spatial-crop-modeler](https://github.com/CGIAR-Climate-Data-Hub/skills/blob/main/.agents/skills/spatial-crop-modeler/SKILL.md)
  · [troubleshooting](https://github.com/CGIAR-Climate-Data-Hub/skills/blob/main/.agents/skills/spatial-crop-modeler/references/troubleshooting.md)
  · [ag-cube-cm](https://github.com/CGIAR-Climate-Data-Hub/ag-cube-cm)
  · [aggeodata](https://github.com/CGIAR-Climate-Data-Hub/aggeodata)
- **More skills**: [CGIAR-Climate-Data-Hub/skills](https://github.com/CGIAR-Climate-Data-Hub/skills)
